In [2]:
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog

SEASON = "2025-26"
SEASON_TYPE = "Regular Season"

team = leaguegamelog.LeagueGameLog(season = SEASON, season_type_all_star=SEASON_TYPE,  player_or_team_abbreviation="T").get_data_frames()[0]
player = leaguegamelog.LeagueGameLog(season = SEASON, season_type_all_star=SEASON_TYPE, player_or_team_abbreviation="P").get_data_frames()[0]

# display(
#     player[player["PTS"] >= 40]
#     .filter(items=["GAME_DATE", "PLAYER_NAME", "MATCHUP", "PTS", "AST", "REB"])
#     .sort_values("PTS", ascending=False)
# )

In [ ]:
# Remove duplicates
team = team.drop_duplicates()
player = player.drop_duplicates()

In [ ]:
# Convert game date from strings into dates
team["GAME_DATE"] = pd.to_datetime(team["GAME_DATE"])
player["GAME_DATE"] = pd.to_datetime(player["GAME_DATE"])

In [ ]:
# Add is home game
team["IS_HOME"] = team["MATCHUP"].str.contains("vs.")
player["IS_HOME"] = player["MATCHUP"].str.contains("vs.")

In [ ]:
# Opponent points
team["OPP_PTS"] = team["PTS"] - team["PLUS_MINUS"]
team[["GAME_DATE", "MATCHUP", "IS_HOME", "PTS", "OPP_PTS"]].head()

In [ ]:
# Validation: each GAME_ID must have exactly 2 rows
counts = team.groupby("GAME_ID").size()
print(counts.value_counts())

assert (counts == 2).all(), "Some GAME_IDs do not have 2 rows!"
print("OK – Total games:", team["GAME_ID"].nunique())

In [ ]:
# Check win percentages when at home
home = team[team["IS_HOME"]]
(home["WL"] == "W").mean()

In [ ]:
# Top 10 players with highest PPG (minimum 50 games played)
(
    player.groupby("PLAYER_ID")
          .filter(lambda g: g["GAME_ID"].nunique() >= 50)   # keep only players with 50+ games
          .groupby(["PLAYER_ID", "PLAYER_NAME"])["PTS"]
          .mean()
          .sort_values(ascending=False)
          .round(1)
          .head(10)
)

In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

engine = create_engine(os.getenv("DATABASE_URL"))

team.columns = team.columns.str.lower()
player.columns = player.columns.str.lower()

team.to_sql("team_games", engine, if_exists="replace", index=False)
player.to_sql("player_games", engine, if_exists="replace", index=False)

print("Saved!")

Saved!
